# Практическое занятие № 13 — регрессия и качество прогноза времени доставки
**ТюмГУ · Технологии искусственного интеллекта — 1 · 90 минут**

**Цель:** перевести задачу с классификации `late` на регрессию `actual_minutes`, обучить базовое, линейное и нелинейное решения, оценить MAE, RMSE, R² и графики остатков.

**Важно:** прогноз строится **в момент оформления заказа**. Поля `late`, `actual_minutes`, `order_id` запрещены внутри X. Предыдущая практика прогнозировала вероятность задержки, теперь целевая переменная — фактические минуты.

**Источник:** `orders_dirty.csv`, используемый на занятиях 3–12. Если недоступен, есть отдельно помеченный демонстрационный CSV. Синтетические результаты не являются результатами на реальном наборе студентов.

## 1. Загрузка, аудит целевой переменной и временной split (0–23 мин)
Запустите ячейку и объясните все исключения. Учебные диапазоны из словаря и неполные метки не равнозначны доказанной ошибке; здесь применяется явная учебная политика. До последнего шага занятия 14 **не оценивайте test**.

In [ ]:
# ОБЩАЯ ПОДГОТОВКА: выполняется самостоятельно в каждом блокноте.
# В Colab загрузите orders_dirty.csv через боковую панель «Файлы».
# При отсутствии официального файла используйте только demo_orders_13_14.csv
# и ОТМЕТЬТЕ В ОТЧЁТЕ, что результаты демонстрационные.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_PATH = Path('orders_dirty.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('demo_orders_13_14.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Загрузите orders_dirty.csv (учебный источник) либо demo_orders_13_14.csv (только демонстрация).')
source_type = 'ОФИЦИАЛЬНЫЙ/УЧЕБНЫЙ НАБОР' if DATA_PATH.name=='orders_dirty.csv' else 'ДЕМОНСТРАЦИОННЫЕ ДАННЫЕ — НЕ СДАВАТЬ КАК ОСНОВНОЙ ИСТОЧНИК'
print('Источник:', DATA_PATH.name, '|', source_type)
raw = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
REQUIRED = {'order_id','created_at','distance_km','queue_size','courier_load','promised_minutes','zone','weather','actual_minutes','late'}
assert REQUIRED.issubset(raw.columns), f'Отсутствуют поля: {sorted(REQUIRED-set(raw.columns))}'
print('Исходный размер:', raw.shape, 'Полных повторов:', int(raw.duplicated().sum()))

# Для регрессии actual_minutes — цель; late запрещено использовать как признак.
# Пустой/нечисловой target не подходит для fit и оценки; диапазон 5–240 —
# учебное правило качества, согласованное со словарём данных, а не физический закон.
df=raw.drop_duplicates().copy()
df['created_at']=pd.to_datetime(df['created_at'], errors='coerce')
df['actual_minutes']=pd.to_numeric(df['actual_minutes'], errors='coerce')
valid_target=df['actual_minutes'].between(5,240,inclusive='both')
valid_time=df['created_at'].notna()
valid_id=df['order_id'].notna()
print('Цель вне 5–240 или NaN:',int((~valid_target).sum()),
      'невалидных дат:',int((~valid_time).sum()),'пустых ID:',int((~valid_id).sum()))
df=df.loc[valid_target & valid_time & valid_id].copy()
# Если разные строки имеют один ID, студент обязан разобраться с ним до разбиения.
assert df['order_id'].is_unique, 'Конфликтующие order_id: разберите вручную до продолжения.'
df=df.sort_values(['created_at','order_id'],kind='stable').reset_index(drop=True)
assert len(df)>=45, f'Для этой демонстрации нужно хотя бы 45 валидных заказов, найдено {len(df)}'
n=len(df)
n_train=int(n*.6); n_dev=int(n*.8)
train=df.iloc[:n_train].copy(); val=df.iloc[n_train:n_dev].copy(); test=df.iloc[n_dev:].copy()
assert len(train)>0 and len(val)>0 and len(test)>0
assert set(train['order_id']).isdisjoint(val['order_id'])
assert set(train['order_id']).isdisjoint(test['order_id'])
assert set(val['order_id']).isdisjoint(test['order_id'])
assert train['created_at'].max()<=val['created_at'].min()<=test['created_at'].min()
print('После очистки:',n,'train/val/test:',list(map(len,[train,val,test])))
print('Временные диапазоны:')
for name,part in [('train',train),('val',val),('test (не открывать до занятия 14!)',test)]:
    print(name, len(part),str(part.created_at.min()),'…',str(part.created_at.max()))

NUM_BASE=['distance_km','queue_size','courier_load','promised_minutes']
CAT=['zone','weather']
DERIVED=['hour_sin','hour_cos','planned_speed_kmh']
NUM=NUM_BASE+DERIVED
INPUT=NUM_BASE+CAT+['created_at']

def make_features(X):
    # Функция НЕ обучается по данным и применяется идентично внутри каждого fold.
    f=X.loc[:,INPUT].copy()
    for k in NUM_BASE:
        f[k]=pd.to_numeric(f[k], errors='coerce')
        f.loc[f[k]<0,k]=np.nan
    f.loc[f['promised_minutes']<=0,'promised_minutes']=np.nan
    for k in CAT:
        s=f[k].astype('string').str.strip()
        s=s.mask(s.str.lower().isin(['','nan','none','null','нет данных','неизвестно']))
        f[k]=s.astype(object).where(s.notna(),np.nan)
    ts=pd.to_datetime(f['created_at'],errors='coerce')
    hour=ts.dt.hour.astype(float)
    f['hour_sin']=np.sin(2*np.pi*hour/24)
    f['hour_cos']=np.cos(2*np.pi*hour/24)
    f['planned_speed_kmh']=60*f['distance_km']/f['promised_minutes']
    f['planned_speed_kmh']=f['planned_speed_kmh'].replace([np.inf,-np.inf],np.nan)
    return f.drop(columns=['created_at'])

num_pipe=Pipeline([('impute',SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True)),('scale',StandardScaler())])
cat_pipe=Pipeline([('impute',SimpleImputer(strategy='constant',fill_value='неизвестно')),
                   ('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))])
prep=ColumnTransformer([('num',num_pipe,NUM),('cat',cat_pipe,CAT)],remainder='drop')

def build_pipeline(regressor):
    return Pipeline([('derive',FunctionTransformer(make_features,validate=False)),
                     ('prep',clone(prep)),('model',regressor)])

def eval_regression(y_true,y_pred):
    return {'MAE':float(mean_absolute_error(y_true,y_pred)),
            'RMSE':float(np.sqrt(mean_squared_error(y_true,y_pred))),
            'R2':float(r2_score(y_true,y_pred))}
Xtrain=train[INPUT].copy(); ytrain=train['actual_minutes'].copy()
Xval=val[INPUT].copy(); yval=val['actual_minutes'].copy()
# test intentionally saved but not used before final check in 14.
assert {'order_id','late','actual_minutes'}.isdisjoint(Xtrain.columns)
print('Входные поля:',INPUT,'Цель: actual_minutes (мин)')

## 2. Метрики и базовая регрессионная модель (23–40 мин)

Определения: **MAE** = (1/n)Σ|yᵢ−ŷᵢ|; **RMSE** = √[(1/n)Σ(yᵢ−ŷᵢ)²]; **R²** = 1−Σ(yᵢ−ŷᵢ)²/Σ(yᵢ−ȳ)². При константном y R² может быть неопределён; здесь предполагается ненулевое разнообразие цели.

Объясните, почему RMSE сильнее реагирует на крупные ошибки, а DummyRegressor обязателен для осмысленного сравнения.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

REGRESSORS={
 'Dummy (median)':DummyRegressor(strategy='median'),
 'Ridge (alpha=1)':Ridge(alpha=1.0),
 'RandomForest':RandomForestRegressor(n_estimators=120,max_depth=8,
                  min_samples_leaf=3,random_state=42,n_jobs=-1)
}
rows=[]; fitted={}; pred_val={}
for name, reg in REGRESSORS.items():
    m=build_pipeline(reg)
    m.fit(Xtrain,ytrain)  # Все fit-трансформации выполняются только на train!
    yhat=m.predict(Xval)
    fitted[name]=m; pred_val[name]=yhat
    rows.append({'model':name,**eval_regression(yval,yhat)})
results=pd.DataFrame(rows).sort_values('MAE').reset_index(drop=True)
print('Результаты на validation (не test):')
display(results.round(3))
assert np.isfinite(results[['MAE','RMSE','R2']].to_numpy()).all()
assert results['MAE'].ge(0).all()
results.to_csv('13_validation_models.csv',index=False,encoding='utf-8-sig')

## 3. Визуализация прогнозов и остатков (40–65 мин)

Остаток `eᵢ = yᵢ − ŷᵢ`. Положительный остаток означает, что модель недооценила время доставки. Сравните точки с диагональю `ŷ=y`, затем посмотрите, нет ли структуры в остатках. Отсутствие явного рисунка **не доказывает** корректность модели.

In [ ]:
best_name=str(results.iloc[0]['model'])
yhat=pred_val[best_name]
residuals=(yval.to_numpy()-yhat)
fig, axes=plt.subplots(1,2,figsize=(11.2,4.1))
axes[0].scatter(yval,yhat,alpha=.75,s=22)
lo=min(float(np.min(yval)),float(np.min(yhat))); hi=max(float(np.max(yval)),float(np.max(yhat)))
axes[0].plot([lo,hi],[lo,hi],'--',label='идеальный прогноз')
axes[0].set(xlabel='Факт, мин',ylabel='Прогноз, мин',title=f'Факт vs прогноз: {best_name}')
axes[0].legend()
axes[1].scatter(yhat,residuals,alpha=.75,s=22)
axes[1].axhline(0,ls='--')
axes[1].set(xlabel='Прогноз, мин',ylabel='Остаток = факт − прогноз, мин',title='Ошибки относительно прогноза')
fig.tight_layout();fig.savefig('13_regression_diagnostics.png',dpi=160);plt.show()

err=pd.DataFrame({'order_id':val['order_id'].values,'actual_minutes':yval.values,
  'predicted_minutes':yhat,'residual_minutes':residuals,'absolute_error':np.abs(residuals)})
display(err.sort_values('absolute_error',ascending=False).head(7).round(2))
err.to_csv('13_validation_predictions.csv',index=False,encoding='utf-8-sig')

## 4. Анализ ошибок по группам (65–79 мин)

Проверьте, одинаково ли модель ошибается в разных зонах или при погодных условиях. У групп с малым n не делайте обобщений. Прогнозы здесь построены моделью, которая не обучалась на validation.

In [ ]:
errors=val[['zone','weather','order_id']].copy()
errors['absolute_error']=np.abs(residuals)
group_stats=errors.groupby('weather',dropna=False)['absolute_error'].agg(['size','mean','median']).rename(columns={'size':'n','mean':'MAE','median':'median_AE'})
display(group_stats.round(2))
group_stats.to_csv('13_errors_by_weather.csv',encoding='utf-8-sig')

## 5. Самостоятельная часть и сдача (79–90 мин)

1. Объясните бизнес-смысл прогноза `actual_minutes`; назовите три запрещённых входа и причину.
2. Сравните Dummy, Ridge и Forest по MAE/RMSE/R²; укажите размеры train/validation.
3. По графикам укажите минимум два наблюдения об остатках с числовыми доказательствами.
4. В качестве дополнительного эксперимента измените `alpha` Ridge и `min_samples_leaf` леса **без просмотра test**; запишите отличие. Для систематического подбора используйте занятие 14.
5. В отчёте сформулируйте два ограничения: синтетичность/дефекты источника, зависимость оценки от времени и размера выборки.

**Сдать:** `Группа_Фамилия_З13_Отчёт.docx` (или PDF), `13_student.ipynb`, CSV с результатами, PNG диагностики. Объём 3–5 страниц без приложений. Перезапустите ядро и выполните все ячейки по порядку.